In [ ]:

import zipfile, tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2

# ── Unzip ─────────────────────────────────────────────────────────────────────
with zipfile.ZipFile("drive/MyDrive/constant_training.zip") as z:
    z.extractall("data")

# ── Dataset ───────────────────────────────────────────────────────────────────
IMG_SIZE, BATCH = 224, 32

def load(split):
    return tf.keras.utils.image_dataset_from_directory(
        "data", labels="inferred", label_mode="categorical",
        class_names=["LOW", "MEDIUM", "HIGH"],   # alphabetical → 0, 1, 2
        image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH,
        validation_split=0.2, subset=split, seed=42,
        color_mode="grayscale"
    )

prep = lambda x, y: (tf.repeat(x, 3, axis=-1) / 127.5 - 1, y)
train_ds = load("training").map(prep)
val_ds   = load("validation").map(prep)

# ── Model ─────────────────────────────────────────────────────────────────────
base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
base.trainable = False

inputs  = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3))
x       = base(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dropout(0.3)(x)
x       = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(3, activation="softmax")(x)
model   = Model(inputs, outputs)

model.compile("adam", "categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=10)

base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(tf.keras.optimizers.Adam(1e-5), "categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=10)

model.save("drive/MyDrive/rf_classifier.h5")
print("Done — classes: LOW=0 MEDIUM=1 HIGH=2")

Found 35000 files belonging to 3 classes.
Using 28000 files for training.
Found 35000 files belonging to 3 classes.
Using 7000 files for validation.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 86s 76ms/step - accuracy: 0.8296 - loss: 0.4217 - val_accuracy: 0.8603 - val_loss: 0.3516
Epoch 2/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 48s 54ms/step - accuracy: 0.8592 - loss: 0.3537 - val_accuracy: 0.8671 - val_loss: 0.3375
Epoch 3/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 48s 55ms/step - accuracy: 0.8695 - loss: 0.3283 - val_accuracy: 0.8683 - val_loss: 0.3323
Epoch 4/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 48s 54ms/step - accuracy: 0.8780 - loss: 0.3074 - val_accuracy: 0.8593 - val_loss: 0.3679
Epoch 5/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 81s 53ms/step - accuracy: 0.8840 - loss: 0.2933 - val_accuracy: 0.8777 - val_loss: 0.3271
Epoch 6/10
875/875 ━━━━━━━━━━━━━━━━━━━━ 47s 54ms/step - accuracy: 0.8904 - loss: 0.2771 - val_accuracy: 0.8726 - val_loss: 0.3343
Epoch 7/10
875/875 ━━━

Done — classes: LOW=0 MEDIUM=1 HIGH=2


In [ ]:
import numpy as np
import tensorflow as tf
from PIL import Image

CLASS_NAMES = ["LOW", "MEDIUM", "HIGH"]
N_VALUES    = [2.2, 3.0, 3.8]

model = tf.keras.models.load_model("drive/MyDrive/rf_classifier.keras")

def predict(image_path):
    img = Image.open(image_path).convert("L").resize((224, 224))
    x   = np.array(img)                  # (224, 224)
    x   = np.stack([x, x, x], axis=-1)  # (224, 224, 3)
    x   = x / 127.5 - 1                 # normalize to [-1, 1]
    x   = np.expand_dims(x, 0)          # (1, 224, 224, 3)

    probs = model.predict(x)[0]
    idx   = np.argmax(probs)
    print(f"Class: {CLASS_NAMES[idx]}  |  N: {N_VALUES[idx]}  |  Confidence: {probs[idx]:.2%}")
    return CLASS_NAMES[idx], N_VALUES[idx]

predict("img1.png")

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
Class: MEDIUM  |  N: 3.0  |  Confidence: 100.00%


('MEDIUM', 3.0)

In [ ]:
import tensorflow as tf, keras
print(tf.__version__, keras.__version__)


2.20.0 3.13.2


In [ ]:
predict("img1.png")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 361ms/step
Class: MEDIUM  |  N: 3.0  |  Confidence: 75.66%


('MEDIUM', 3.0)